# LAB 09 - Tokenization và label alignment cho NER

Dataset CoNLL-2003 được gán nhãn theo **word**, còn Transformer làm việc trên **subword token**. Notebook này xem tokenizer `distilbert-base-cased` tách word như thế nào và gán nhãn BIO cho từng token ra sao (label alignment).

Model, tokenizer và dataset được load offline từ `../hf_models`.

In [1]:
import os
os.environ["HF_HOME"] = os.path.abspath("../hf_models")          # đặt trước khi import transformers
os.environ["HF_HUB_OFFLINE"] = "1"

import numpy as np
import pandas as pd
from transformers import AutoTokenizer

from ner_data import LABEL2ID, LABELS, load_conll

MODEL_NAME = "distilbert/distilbert-base-cased"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)
wid = lambda enc, b=0: ["-" if w is None else w for w in enc.word_ids(b)]
train, test = load_conll("train"), load_conll("test")
print("vocab:", tok.vocab_size, "| train / test câu:", len(train), len(test))
print("nhãn:", LABELS)

vocab: 28996 | train / test câu: 14041 3453
nhãn: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


## 1. Tokenize input đã tách sẵn thành word

`is_split_into_words=True` báo cho tokenizer biết input là list các word, nên tokenizer giữ được thông tin token nào thuộc word nào (`word_ids`).

In [2]:
batch_words = [["Elon", "Musk", "founded", "SpaceX", "in", "California", "."],
               ["Barack", "Obama", "visited", "Paris", "."]]
tokens = tok(batch_words, is_split_into_words=True, truncation=True, padding=True)
print("keys:", list(tokens.keys()))
for b in range(2):
    print(f"\n--- câu {b}")
    print(pd.DataFrame({"token": tokens.tokens(b), "input_id": tokens["input_ids"][b],
                        "attention_mask": tokens["attention_mask"][b], "word_id": wid(tokens, b)}).T.to_string(header=False))

keys: ['input_ids', 'token_type_ids', 'attention_mask']

--- câu 0
token           [CLS]    El  ##on     Mu  ##sk  founded  Space   ##X    in  California    .  [SEP]
input_id          101  2896  1320  19569  5276     1771   4525  3190  1107        1756  119    102
attention_mask      1     1     1      1     1        1      1     1     1           1    1      1
word_id             -     0     0      1     1        2      3     3     4           5    6      -

--- câu 1
token           [CLS]  Barack  Obama  visited  Paris    .  [SEP]  [PAD]  [PAD]  [PAD]  [PAD]  [PAD]
input_id          101   14319   7661     3891   2123  119    102      0      0      0      0      0
attention_mask      1       1      1        1      1    1      1      0      0      0      0      0
word_id             -       0      1        2      3    4      -      -      -      -      -      -


- `word_ids`: token thứ $i$ thuộc word nào. Giá trị `None` là special token ([CLS], [SEP]) hoặc padding.
- Câu 2 ngắn hơn nên được pad: các vị trí `[PAD]` có `attention_mask = 0` và `word_id = None`.

## 2. Label alignment: Strategy A (chọn dùng) và Strategy B (để so sánh)

- **Strategy A:** token đầu tiên của word nhận nhãn của word. Các subword `##...`, special token và padding nhận `-100` (IGNORE), nên không tính vào loss.
- **Strategy B:** lan truyền nhãn cho mọi subword. Subword sau của một word có nhãn `B-X` nhận `I-X`.

Lab dùng **Strategy A** cho cả train lẫn evaluate. Lý do:

1. Mỗi word đóng góp đúng một nhãn vào loss, nên tên bị tách thành nhiều mảnh không bị tính trọng số nhiều hơn.
2. Khi dự đoán, chỉ đọc nhãn ở token đầu nên kết quả quay về đúng mức word để so với gold, không cần quy tắc gộp nhiều nhãn thành một.

In [3]:
IGNORE = -100

def align_labels(word_ids, word_labels, strategy="A"):
    out, prev = [], None
    for w in word_ids:
        if w is None:
            out.append(IGNORE)
        elif w != prev:                                    # token đầu của word
            out.append(LABEL2ID[word_labels[w]])
        elif strategy == "A":
            out.append(IGNORE)
        else:                                              # strategy B: B-X → I-X cho subword sau
            out.append(LABEL2ID[word_labels[w].replace("B-", "I-")])
        prev = w
    return out

words = ["Elon", "Musk", "founded", "SpaceX", "in", "California", "."]
labels = ["B-PER", "I-PER", "O", "B-ORG", "O", "B-LOC", "O"]
enc = tok(words, is_split_into_words=True)
name = lambda i: "IGNORE" if i == IGNORE else LABELS[i]
pd.DataFrame({"token": enc.tokens(), "word_id": wid(enc),
              "Strategy A": [name(i) for i in align_labels(enc.word_ids(), labels, "A")],
              "Strategy B": [name(i) for i in align_labels(enc.word_ids(), labels, "B")]}).T

,0,1,2,3,4,5,6,7,8,9,10,11
token,[CLS],El,##on,Mu,##sk,founded,Space,##X,in,California,.,[SEP]
word_id,-,0,0,1,1,2,3,3,4,5,6,-
Strategy A,IGNORE,B-PER,IGNORE,I-PER,IGNORE,O,B-ORG,IGNORE,O,B-LOC,O,IGNORE
Strategy B,IGNORE,B-PER,I-PER,I-PER,I-PER,O,B-ORG,I-ORG,O,B-LOC,O,IGNORE


Một câu thật trong CoNLL train có nhiều word bị tách:

In [4]:
def n_split_entities(s):
    return sum(l != "O" and len(tok.tokenize(w)) > 1 for w, l in zip(*s)) if 8 <= len(s[0]) <= 14 else 0
words, labels = max(train[:3000], key=n_split_entities)
enc = tok(words, is_split_into_words=True)
pd.DataFrame({"token": enc.tokens(), "word_id": wid(enc),
              "Strategy A": [name(i) for i in align_labels(enc.word_ids(), labels, "A")]}).T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29
token,[CLS],In,##zam,##am,-,ul,-,Ha,##q,",",Sal,##im,Malik,",",As,##if,Mu,##j,##ta,##ba,",",Was,##im,A,##kra,##m,",",Mo,##in,[SEP]
word_id,-,0,0,0,0,0,0,0,0,1,2,2,3,4,5,5,6,6,6,6,7,8,8,9,9,9,10,11,11,-
Strategy A,IGNORE,B-PER,IGNORE,IGNORE,IGNORE,IGNORE,IGNORE,IGNORE,IGNORE,O,B-PER,IGNORE,I-PER,O,B-PER,IGNORE,I-PER,IGNORE,IGNORE,IGNORE,O,B-PER,IGNORE,I-PER,IGNORE,IGNORE,O,B-PER,IGNORE,IGNORE


### Kiểm tra alignment trên toàn bộ train

Với Strategy A, số vị trí có nhãn phải bằng đúng số word, và đọc lại nhãn ở token đầu phải ra đúng chuỗi nhãn gốc.

In [5]:
enc_all = tok([w for w, _ in train], is_split_into_words=True, truncation=True, max_length=128)
n_ok = n_trunc = n_tok = n_ign = 0
lab_tokens_B = 0
for b, (words, labels) in enumerate(train):
    wid = enc_all.word_ids(b)
    A = align_labels(wid, labels, "A")
    back = [LABELS[A[i]] for i in range(len(wid)) if A[i] != IGNORE]
    if back == labels:
        n_ok += 1
    elif max(w for w in wid if w is not None) + 1 < len(words):
        n_trunc += 1
    n_tok += len(wid)
    n_ign += sum(a == IGNORE for a in A)
    lab_tokens_B += sum(a != IGNORE for a in align_labels(wid, labels, "B"))
n_words = sum(len(w) for w, _ in train)
print(f"câu khôi phục đúng nhãn gốc: {n_ok}/{len(train)} | câu bị cắt ở 128 token: {n_trunc}")
print(f"token: {n_tok:,} | IGNORE: {n_ign:,} ({100 * n_ign / n_tok:.1f}%) | word: {n_words:,}")
print(f"vị trí tính loss - Strategy A: {n_tok - n_ign:,} (= số word) | Strategy B: {lab_tokens_B:,}")

câu khôi phục đúng nhãn gốc: 14040/14041 | câu bị cắt ở 128 token: 1
token: 300,632 | IGNORE: 97,039 (32.3%) | word: 203,621
vị trí tính loss - Strategy A: 203,593 (= số word) | Strategy B: 272,550


## 3. Experiment 1 - Tokenization analysis

### 24 câu từ test set: 12 câu ngắn (≤ 8 word) và 12 câu dài (≥ 25 word)

- **tokens**: số subword token, không tính [CLS] và [SEP].
- **split words**: số word bị tách thành từ 2 token trở lên.
- $R = \dfrac{\text{số subword token}}{\text{số word}}$

In [6]:
def sent_stats(words, labels):
    pieces = [len(tok.tokenize(w)) for w in words]
    return {"words": len(words), "tokens": sum(pieces), "split words": sum(p > 1 for p in pieces),
            "R": sum(pieces) / len(words), "entities": sum(l.startswith("B-") for l in labels)}

rng = np.random.default_rng(42)
short = [s for s in test if 4 <= len(s[0]) <= 8]
long_ = [s for s in test if len(s[0]) >= 25]
sample = [("ngắn", short[i]) for i in rng.choice(len(short), 12, replace=False)] + \
         [("dài", long_[i]) for i in rng.choice(len(long_), 12, replace=False)]
rows = [{"group": g, "text": " ".join(w)[:70], **sent_stats(w, l)} for g, (w, l) in sample]
exp1 = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 80)
exp1.round(2)

,group,text,words,tokens,split words,R,entities
0,ngắn,"St. Louis 253,821 223,172",4,10,3,2.50,1
1,ngắn,M. Waugh c Murray b Benjamin 27,7,9,2,1.29,3
2,ngắn,Albion 16 6 6 4 21 17 24,8,8,0,1.00,1
3,ngắn,8. Florence Masnada ( France ) 1:18.31,7,15,3,2.14,2
4,ngắn,( 52.82 / 53.13 ),5,9,2,1.80,0
5,ngắn,Sunderland 1 Wimbledon 3,4,4,0,1.00,2
6,ngắn,M. Bevan st Murray b Hooper 3,7,10,3,1.43,3
7,ngắn,""" Weather forecasts have been sketchy .",7,9,2,1.29,0
8,ngắn,"HOBART , Australia 1996-12-06",4,11,2,2.75,2
9,ngắn,2. Kusunose Shiho ( Japan ) 1:24.77,7,15,4,2.14,2


In [7]:
exp1.groupby("group")[["words", "tokens", "split words", "R"]].mean().round(2)

,words,tokens,split words,R
group,,,,
dài,31.58,35.42,2.92,1.12
ngắn,5.67,10.00,2.08,1.86


### Thống kê trên toàn bộ train (203K word)

In [8]:
def kind(w):
    if not any(c.isalnum() for c in w):
        return "dấu câu"
    if any(c.isdigit() for c in w):
        return "có chữ số"
    if w.isupper() and len(w) > 1:
        return "VIẾT HOA HẾT"
    if w[0].isupper():
        return "Viết hoa chữ đầu"
    return "chữ thường"

rows = []
for words, labels in train:
    caps_sentence = sum(w.isupper() and len(w) > 1 for w in words) / len(words) > 0.5      # headline viết hoa toàn bộ
    for w, l in zip(words, labels):
        n = len(tok.tokenize(w))
        rows.append({"word": w, "label": l, "type": l[2:] if l != "O" else "O", "entity": l != "O",
                     "kind": kind(w), "pieces": n, "headline": caps_sentence, "sent_len": len(words)})
W = pd.DataFrame(rows)
W["split"] = W.pieces > 1
W["sent"] = np.where(W.sent_len <= 8, "câu ngắn (≤ 8)", np.where(W.sent_len >= 25, "câu dài (≥ 25)", "trung bình"))

def summary(by):
    return W.groupby(by).agg(words=("pieces", "size"), R=("pieces", "mean"), pct_split=("split", "mean")).assign(
        pct_split=lambda d: 100 * d.pct_split).round(3)

print("Entity vs non-entity"); display(summary("entity"))
print("Theo loại entity"); display(summary("type"))
print("Theo độ dài câu"); display(summary("sent"))
print("Theo dạng word"); display(summary("kind"))

Entity vs non-entity


,words,R,pct_split
entity,,,
False,169578,1.224,11.176
True,34043,1.910,45.081


Theo loại entity


,words,R,pct_split
type,,,
LOC,8297,1.812,31.108
MISC,4593,1.724,32.833
O,169578,1.224,11.176
ORG,10025,1.943,47.092
PER,11128,2.031,58.744


Theo độ dài câu


,words,R,pct_split
sent,,,
câu dài (≥ 25),101080,1.182,10.566
câu ngắn (≤ 8),29560,1.921,38.068
trung bình,72981,1.320,16.944


Theo dạng word


,words,R,pct_split
kind,,,
VIẾT HOA HẾT,7412,3.317,90.165
Viết hoa chữ đầu,37405,1.559,31.384
chữ thường,113764,1.100,6.583
có chữ số,19159,1.972,41.307
dấu câu,25881,1.037,1.831


In [9]:
print("Word thường bị tách nhiều nhất (xuất hiện ≥ 3 lần):")
top = W.groupby("word").agg(n=("pieces", "size"), pieces=("pieces", "first"), label=("label", lambda s: s.mode()[0]))
top = top[top.n >= 3].sort_values("pieces", ascending=False).head(15)
top.assign(tokens=[" ".join(tok.tokenize(w)) for w in top.index])

Word thường bị tách nhiều nhất (xuất hiện ≥ 3 lần):


,n,pieces,label,tokens
word,,,,
------------------------,7,24,O,- - - - - - - - - - - - - - - - - - - - - - - -
SPA-FRANCORCHAMPS,3,10,B-LOC,SP ##A - F ##RA ##NC ##OR ##CH ##AM ##PS
Ata-ur-Rehman,5,8,B-PER,At ##a - u ##r - Re ##hman
PHILADELPHIA,17,8,B-ORG,P ##H ##IL ##AD ##EL ##P ##H ##IA
312-408-8787,3,8,O,312 - 40 ##8 - 87 ##8 ##7
Inzamam-ul-Haq,12,8,B-PER,In ##zam ##am - ul - Ha ##q
JOHANNESBURG,12,7,B-LOC,J ##OH ##AN ##NE ##SB ##UR ##G
"12,900-13,100",3,7,O,"12 , 900 - 13 , 100"
Moslem-Croat,3,7,B-MISC,Mo ##sle ##m - C ##roa ##t


In [10]:
print("Cùng một từ, viết thường / viết hoa chữ đầu / viết hoa hết:")
pd.DataFrame([{"word": w, **{f: " ".join(tok.tokenize(x)) for f, x in
               [("lower", w.lower()), ("Title", w.title()), ("UPPER", w.upper())]}}
              for w in ["germany", "washington", "manchester", "results", "soccer"]])

Cùng một từ, viết thường / viết hoa chữ đầu / viết hoa hết:


,word,lower,Title,UPPER
0,germany,g ##erman ##y,Germany,GE ##R ##MA ##N ##Y
1,washington,washing ##ton,Washington,WA ##S ##H ##ING ##TO ##N
2,manchester,man ##chester,Manchester,MA ##NC ##H ##ES ##TE ##R
3,results,results,Results,R ##ES ##U ##LT ##S
4,soccer,soccer,Soccer,S ##OC ##CE ##R


**Nhận xét Experiment 1**

| | R (token / word) | % word bị tách |
|---|---|---|
| 24 câu: câu ngắn / câu dài | 1.86 / 1.12 | |
| Train: câu ≤ 8 word / câu ≥ 25 word | 1.92 / 1.18 | 38.1 / 10.6 |
| Train: entity / non-entity | 1.91 / 1.22 | 45.1 / 11.2 |
| PER / ORG / LOC / MISC | 2.03 / 1.94 / 1.81 / 1.72 | 58.7 / 47.1 / 31.1 / 32.8 |

- **Câu ngắn bị tách nhiều hơn câu dài.** Câu ngắn trong CoNLL thường là tiêu đề, dateline hoặc bảng kết quả thể thao ("HOBART , Australia 1996-12-06", "NY RANGERS AT TORONTO"), gồm toàn tên riêng, số và chữ in hoa. Câu dài là văn xuôi với nhiều từ thông dụng.
- **Entity bị tách nhiều gấp 4 lần non-entity.** Tên người bị tách nhiều nhất (58.7%), vì tên riêng hiếm khi có trong vocabulary 29K token.
- **Những loại từ thường bị tách:**
  - Từ VIẾT HOA HẾT: R = 3.32, 90% bị tách.
  - Từ có chữ số: R = 1.97, 41% bị tách.
  - Tên riêng hiếm và tên có gạch nối ("Inzamam-ul-Haq" thành 8 token).
  - Ngược lại, từ thường viết chữ thường chỉ có 6.6% bị tách.
- **Tokenizer cased phân biệt chữ hoa:** "Germany" là một token, nhưng "GERMANY" thành 5 mảnh vô nghĩa (`GE ##R ##MA ##N ##Y`). CoNLL có nhiều tiêu đề in hoa toàn bộ, nên đây là nơi dự kiến model dễ sai hơn.

## 4. Ảnh hưởng của Strategy A

Word bị tách chỉ có token đầu mang nhãn. Thống kê dưới đây cho biết bao nhiêu entity mà phần lớn token không có nhãn trực tiếp.

In [11]:
ent = W[W.entity]
print(f"word thuộc entity bị tách: {100 * ent.split.mean():.1f}% (non-entity: {100 * W[~W.entity].split.mean():.1f}%)")
print(f"token của entity word: {ent.pieces.sum():,}, trong đó có nhãn (token đầu): {len(ent):,} "
      f"→ {100 * (1 - len(ent) / ent.pieces.sum()):.1f}% là IGNORE")

word thuộc entity bị tách: 45.1% (non-entity: 11.2%)
token của entity word: 65,030, trong đó có nhãn (token đầu): 34,043 → 47.7% là IGNORE


**Nhận xét về Strategy A**

- Alignment đúng: đọc lại nhãn ở token đầu khôi phục đúng 14,040/14,041 câu. Câu còn lại dài hơn 128 token và bị cắt.
- 32.3% token là IGNORE (special token và subword). Số vị trí tính loss bằng đúng số word.
- Nếu dùng Strategy B, số vị trí tính loss là 272,550, nhiều hơn 34%. Phần tăng thêm dồn vào entity vì entity bị tách nhiều: một tên như "Inzamam-ul-Haq" sẽ được tính 8 lần, trong khi "Smith" chỉ được tính 1 lần.
- Cái giá của Strategy A: 47.7% token thuộc entity không có nhãn trực tiếp. Quyết định dồn vào token đầu, mà token đầu nhiều khi là mảnh ngắn vô nghĩa ("In", "P", "J"). Token đầu vẫn nhìn thấy các subword sau qua self-attention, nhưng với tên in hoa toàn bộ, thông tin ở token đầu rất ít. Notebook fine-tuning sẽ kiểm tra lỗi theo số subword của entity.